In [1]:
%pip install ultralytics filterpy lap -q
from google.colab import files
import os

print("Please upload your custom 'best.pt' model file.")
uploaded = files.upload()

if not uploaded:
    print("\nNo model file was uploaded. Please run the cell again.")
else:
    model_path = list(uploaded.keys())[0]
    print(f"\nUploaded '{model_path}' successfully.")
    # Store the path for later use
    %store model_path

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━

Saving best.pt to best.pt

Uploaded 'best.pt' successfully.
Stored 'model_path' (str)


In [2]:
#Link for the Sample Video
#https://drive.google.com/file/d/12v1Eco15GUhoPMOEZZBIXcae2aShCmNO/view?usp=sharing
from google.colab import files

print("Please upload your video file.")
uploaded = files.upload()

if not uploaded:
    print("\nNo video file was uploaded. Please run the cell again.")
else:
    video_path = list(uploaded.keys())[0]
    print(f"\nUploaded '{video_path}' successfully.")
    # Store the path for later use
    %store video_path

Please upload your video file.


Saving 20250609_091016 (1).mp4 to 20250609_091016 (1).mp4

Uploaded '20250609_091016 (1).mp4' successfully.
Stored 'video_path' (str)


In [ ]:
%pip install ultralytics filterpy lap -q

# --- SORT Algorithm Code ---
from __future__ import print_function
import numpy as np
from filterpy.kalman import KalmanFilter
import time

def linear_assignment(cost_matrix):
  try:
    import lap
    _, x, y = lap.lapjv(cost_matrix, extend_cost=True)
    return np.array([[y[i],i] for i in x if i >= 0])
  except ImportError:
    from scipy.optimize import linear_sum_assignment
    x, y = linear_sum_assignment(cost_matrix)
    return np.array(list(zip(x, y)))

def iou_batch(bb_test, bb_gt):
  bb_gt = np.expand_dims(bb_gt, 0)
  bb_test = np.expand_dims(bb_test, 1)
  xx1 = np.maximum(bb_test[..., 0], bb_gt[..., 0])
  yy1 = np.maximum(bb_test[..., 1], bb_gt[..., 1])
  xx2 = np.minimum(bb_test[..., 2], bb_gt[..., 2])
  yy2 = np.minimum(bb_test[..., 3], bb_gt[..., 3])
  w = np.maximum(0., xx2 - xx1)
  h = np.maximum(0., yy2 - yy1)
  wh = w * h
  o = wh / ((bb_test[..., 2] - bb_test[..., 0]) * (bb_test[..., 3] - bb_test[..., 1])
    + (bb_gt[..., 2] - bb_gt[..., 0]) * (bb_gt[..., 3] - bb_gt[..., 1]) - wh)
  return(o)

def convert_bbox_to_z(bbox):
  w = bbox[2] - bbox[0]; h = bbox[3] - bbox[1]
  x = bbox[0] + w/2.; y = bbox[1] + h/2.
  s = w * h; r = w / float(h)
  return np.array([x, y, s, r]).reshape((4, 1))

def convert_x_to_bbox(x,score=None):
  w = np.sqrt(x[2] * x[3]); h = x[2] / w
  if(score==None):
    return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.]).reshape((1,4))
  else:
    return np.array([x[0]-w/2.,x[1]-h/2.,x[0]+w/2.,x[1]+h/2.,score]).reshape((1,5))

class KalmanBoxTracker(object):
  count = 0
  def __init__(self,bbox):
    self.kf = KalmanFilter(dim_x=7, dim_z=4)
    self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],[0,0,0,1,0,0,0],  [0,0,0,0,1,0,0],[0,0,0,0,0,1,0],[0,0,0,0,0,0,1]])
    self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],[0,0,1,0,0,0,0],[0,0,0,1,0,0,0]])
    self.kf.R[2:,2:] *= 10.
    self.kf.P[4:,4:] *= 1000.; self.kf.P *= 10.
    self.kf.Q[-1,-1] *= 0.01; self.kf.Q[4:,4:] *= 0.01
    self.kf.x[:4] = convert_bbox_to_z(bbox)
    self.time_since_update = 0; self.id = KalmanBoxTracker.count; KalmanBoxTracker.count += 1
    self.history = []; self.hits = 0; self.hit_streak = 0; self.age = 0

  def update(self,bbox):
    self.time_since_update = 0; self.history = []; self.hits += 1; self.hit_streak += 1
    self.kf.update(convert_bbox_to_z(bbox))

  def predict(self):
    if((self.kf.x[6]+self.kf.x[2])<=0):
      self.kf.x[6] *= 0.0
    self.kf.predict(); self.age += 1
    if(self.time_since_update>0):
      self.hit_streak = 0
    self.time_since_update += 1
    self.history.append(convert_x_to_bbox(self.kf.x))
    return self.history[-1]

  def get_state(self):
    return convert_x_to_bbox(self.kf.x)

def associate_detections_to_trackers(detections,trackers,iou_threshold = 0.3):
  if(len(trackers)==0):
    return np.empty((0,2),dtype=int), np.arange(len(detections)), np.empty((0,5),dtype=int)
  iou_matrix = iou_batch(detections, trackers)
  if min(iou_matrix.shape) > 0:
    a = (iou_matrix > iou_threshold).astype(np.int32)
    if a.sum(1).max() == 1 and a.sum(0).max() == 1:
        matched_indices = np.stack(np.where(a), axis=1)
    else:
      matched_indices = linear_assignment(-iou_matrix)
  else:
    matched_indices = np.empty(shape=(0,2))
  unmatched_detections = []
  for d, det in enumerate(detections):
    if(d not in matched_indices[:,0]):
      unmatched_detections.append(d)
  unmatched_trackers = []
  for t, trk in enumerate(trackers):
    if(t not in matched_indices[:,1]):
      unmatched_trackers.append(t)
  matches = []
  for m in matched_indices:
    if(iou_matrix[m[0], m[1]]<iou_threshold):
      unmatched_detections.append(m[0]); unmatched_trackers.append(m[1])
    else:
      matches.append(m.reshape(1,2))
  if(len(matches)==0):
    matches = np.empty((0,2),dtype=int)
  else:
    matches = np.concatenate(matches,axis=0)
  return matches, np.array(unmatched_detections), np.array(unmatched_trackers)

class Sort(object):
  def __init__(self, max_age=1, min_hits=3, iou_threshold=0.3):
    self.max_age = max_age; self.min_hits = min_hits
    self.iou_threshold = iou_threshold; self.trackers = []
    self.frame_count = 0; KalmanBoxTracker.count = 0

  def update(self, dets=np.empty((0, 5))):
    self.frame_count += 1
    trks = np.zeros((len(self.trackers), 5))
    to_del = []; ret = []
    for t, trk in enumerate(trks):
      pos = self.trackers[t].predict()[0]
      trk[:] = [pos[0], pos[1], pos[2], pos[3], 0]
      if np.any(np.isnan(pos)):
        to_del.append(t)
    trks = np.ma.compress_rows(np.ma.masked_invalid(trks))
    for t in reversed(to_del):
      self.trackers.pop(t)
    matched, unmatched_dets, unmatched_trks = associate_detections_to_trackers(dets,trks, self.iou_threshold)
    for m in matched:
      self.trackers[m[1]].update(dets[m[0], :])
    for i in unmatched_dets:
        trk = KalmanBoxTracker(dets[i,:])
        self.trackers.append(trk)
    i = len(self.trackers)
    for trk in reversed(self.trackers):
        d = trk.get_state()[0]
        if (trk.time_since_update < 1) and (trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits):
          ret.append(np.concatenate((d,[trk.id+1])).reshape(1,-1))
        i -= 1
        if(trk.time_since_update > self.max_age):
          self.trackers.pop(i)
    if(len(ret)>0):
      return np.concatenate(ret)
    return np.empty((0,5))

print("Setup complete and SORT algorithm loaded.")

Setup complete and SORT algorithm loaded.


In [ ]:
import cv2
from ultralytics import YOLO
import torch

# --- Configuration ---
# Your custom class names from your YAML file
CUSTOM_CLASS_NAMES = {
    0: "bat", 1: "bottle", 2: "calender", 3: "car", 4: "clock",
    5: "comb", 6: "cup", 7: "dragon", 8: "far", 9: "groot",
    10: "ironman", 11: "laptop", 12: "light", 13: "mouse"
}

# =====> ACTION REQUIRED: EDIT THIS LIST <=====
# Set which of your custom classes you want to track by their ID number.
# For example, to track only 'car' (ID 3) and 'ironman' (ID 10):
# CLASSES_TO_TRACK = [3, 10]
# To track all your custom objects:
CLASSES_TO_TRACK = list(range(14)) # Tracks all classes from 0 to 13

# Confidence threshold for detection
CONFIDENCE_THRESHOLD = 0.25

# SORT Parameters (you can tune these)
SORT_MAX_AGE = 10
SORT_MIN_HITS = 2
SORT_IOU_THRESHOLD = 0.2

# --- Retrieve uploaded file paths ---
try:
    %store -r model_path
    %store -r video_path
    YOLO_MODEL_PATH = model_path
    INPUT_VIDEO_PATH = video_path
except NameError:
    print("Model or video path not found. Please run Step 1 and 2 first.")
    # Stop execution if files are not available
    exit()

OUTPUT_VIDEO_PATH = "output_tracked.mp4"


# --- Initialization ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load your custom YOLO model
model = YOLO(YOLO_MODEL_PATH)
model.to(device)

# Initialize SORT tracker
mot_tracker = Sort(max_age=SORT_MAX_AGE,
                   min_hits=SORT_MIN_HITS,
                   iou_threshold=SORT_IOU_THRESHOLD)

# Open video file
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video {INPUT_VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

# Generate random colors for each class for consistent coloring
np.random.seed(42)
colors = np.random.randint(0, 255, size=(len(CUSTOM_CLASS_NAMES), 3), dtype="uint8")


# --- Processing Loop ---
frame_num = 0
total_start_time = time.time()
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    start_time = time.time()

    # 1. Run YOLO detection
    results = model(frame, stream=True, verbose=False)
    detections_for_sort = []
    for r in results:
        boxes = r.boxes
        for box in boxes:
            class_id = box.cls[0].int().item()
            if class_id in CLASSES_TO_TRACK:
                if box.conf[0].item() > CONFIDENCE_THRESHOLD:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    score = box.conf[0].cpu().numpy()
                    # Append class_id to the detection to use for labeling
                    detections_for_sort.append([x1, y1, x2, y2, score, class_id])

    # 2. Update SORT tracker
    # We pass only [x1, y1, x2, y2, score] to SORT
    if len(detections_for_sort) > 0:
        detections_np = np.array(detections_for_sort)[:, :5]
        tracked_objects = mot_tracker.update(detections_np)

        # Correlate tracked objects with original detections to get class_id back
        # This is a simple way; for high-speed, a more efficient lookup could be used
        final_tracked_objects = []
        for track in tracked_objects:
            track_box = track[:4]
            best_match_iou = -1
            matched_class_id = -1
            for det in detections_for_sort:
                det_box = det[:4]
                iou = iou_batch(np.array([track_box]), np.array([det_box]))[0][0]
                if iou > best_match_iou:
                    best_match_iou = iou
                    matched_class_id = det[5]
            if matched_class_id != -1:
                final_tracked_objects.append(list(track) + [matched_class_id])
    else:
        tracked_objects = mot_tracker.update()
        final_tracked_objects = []


    # 3. Draw tracked objects
    for obj in final_tracked_objects:
        x1, y1, x2, y2, track_id, class_id = obj
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        track_id = int(track_id)
        class_id = int(class_id)

        color = colors[class_id].tolist()
        class_name = CUSTOM_CLASS_NAMES.get(class_id, "Unknown")
        label = f"{class_name} ID: {track_id}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    out.write(frame)
    print(f"Frame {frame_num}: Processing time: {(time.time() - start_time):.3f}s", end='\r')


# --- Cleanup ---
total_processing_time = time.time() - total_start_time
cap.release()
out.release()
print("\n\n" + "="*50)
print(f"Processing finished in {total_processing_time:.2f} seconds.")
print(f"Average FPS: {frame_num / total_processing_time:.2f}")
print(f"Output video saved to: {OUTPUT_VIDEO_PATH}")
print("="*50)
#OUTPUT VIDEO LINK
#https://drive.google.com/file/d/1Bcldneq6NEkatER_UYPMVIIWG6lBTKly/view?usp=sharing


Using device: cuda


KeyboardInterrupt: 

In [3]:
#Using DeepSort

In [5]:
%pip install ultralytics filterpy lap -q
%pip install deep_sort_realtime scikit-learn -q
import cv2
from ultralytics import YOLO
import torch
import numpy as np
import time
from deep_sort_realtime.deepsort_tracker import DeepSort

# --- Configuration ---
CUSTOM_CLASS_NAMES = {
    0: "bat", 1: "bottle", 2: "calender", 3: "car", 4: "clock",
    5: "comb", 6: "cup", 7: "dragon", 8: "far", 9: "groot",
    10: "ironman", 11: "laptop", 12: "light", 13: "mouse"
}

CLASSES_TO_TRACK = list(range(14))  # Track all classes
CONFIDENCE_THRESHOLD = 0.25

# --- Load paths ---
try:
    %store -r model_path
    %store -r video_path
    YOLO_MODEL_PATH = model_path
    INPUT_VIDEO_PATH = video_path
except NameError:
    print("Model or video path not found. Please run Step 1 and 2 first.")
    exit()

OUTPUT_VIDEO_PATH = "output_deepsort_tracked.mp4"

# --- Device ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 90.5 MB/s eta 0:00:00
Using device: cuda


In [10]:
# Load YOLO model
model = YOLO(YOLO_MODEL_PATH)
model.to(device)

# Initialize Deep SORT tracker
tracker = DeepSort(max_age=30,  # max frames to keep lost tracks
                   n_init=3,    # number of frames before confirmed
                   max_cosine_distance=0.3,
                   nn_budget=100)

# Open video
cap = cv2.VideoCapture(INPUT_VIDEO_PATH)
if not cap.isOpened():
    print(f"Error opening video: {INPUT_VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))

np.random.seed(42)
colors = np.random.randint(0, 255, size=(len(CUSTOM_CLASS_NAMES), 3), dtype="uint8")

frame_num = 0
total_start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1

    # Run YOLO detection
    results = model(frame, stream=True, verbose=False)

    detections = []  # format: [ [x1, y1, x2, y2, score, class_id], ... ]

    for r in results:
        boxes = r.boxes
        for box in boxes:
            class_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            if class_id in CLASSES_TO_TRACK and conf > CONFIDENCE_THRESHOLD:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                detections.append([x1, y1, x2, y2, conf, class_id])

    # Deep SORT expects detections as [[x1,y1,x2,y2,score], ...] (class_id will be passed separately)
    dets_for_tracker = dets_for_tracker = [([d[0], d[1], d[2], d[3]], d[4]) for d in detections]

    tracks = tracker.update_tracks(dets_for_tracker, frame=frame)
    # Optional: You can pass additional class info in `tracker.update_tracks` for labeling
    # But Deep SORT itself only needs bbox+score for tracking

    # Update tracker


    # Draw tracks
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        ltrb = track.to_ltrb()  # left, top, right, bottom
        bbox = list(map(int, ltrb))
        # Find class for this track by matching bbox to original detections (IoU or simple box overlap)
        matched_class = "Unknown"
        max_iou = 0

        for det in detections:
            iou = (
                max(0, min(bbox[2], det[2]) - max(bbox[0], det[0])) *
                max(0, min(bbox[3], det[3]) - max(bbox[1], det[1]))
            )
            area1 = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            area2 = (det[2] - det[0]) * (det[3] - det[1])
            union = area1 + area2 - iou
            iou_score = iou / union if union > 0 else 0
            if iou_score > max_iou:
                max_iou = iou_score
                matched_class = CUSTOM_CLASS_NAMES.get(det[5], "Unknown")

        color = colors[list(CUSTOM_CLASS_NAMES.values()).index(matched_class) if matched_class in CUSTOM_CLASS_NAMES.values() else 0].tolist()
        label = f"{matched_class} ID: {track_id}"

        cv2.rectangle(frame, (bbox[0], bbox[1]), (bbox[2], bbox[3]), color, 2)
        cv2.putText(frame, label, (bbox[0], bbox[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    out.write(frame)
    # Optional: show frame
    # cv2.imshow("Frame", frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
out.release()
# cv2.destroyAllWindows()

print(f"Tracking complete. Output saved to {OUTPUT_VIDEO_PATH}")
#Link of OutPut https://drive.google.com/file/d/1aBZY2ioXS-FWIzdY4uXoY5j6k5DsKVU3/view?usp=sharing

KeyboardInterrupt: 